# DiffSplat — End-to-End Demo Notebook

This notebook walks through the full DiffSplat workflow:

1. **Environment setup** – clone the official repo and install dependencies
2. **Download checkpoints** – pull pretrained weights from Hugging Face
3. **Download benchmark data** – T3Bench prompts and GSO rendered images
4. **Text-conditioned inference** – generate 3D splats from a text prompt
5. **Image-conditioned inference** – reconstruct 3D splats from a reference image
6. **Image quality evaluation** – compute PSNR / SSIM / LPIPS against GT views
7. **Ablation: render-loss comparison** – compare render vs no-render checkpoints

> **GPU requirement**: a CUDA-capable GPU with ≥ 16 GB VRAM is strongly recommended.
> The SD1.5 model runs comfortably on 24 GB; SD3.5-M needs ~40 GB.

## 0 — Configuration

Edit the variables below to match your setup before running anything else.

In [ ]:
import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = Path().resolve()          # directory containing this notebook
REPO_DIR      = NOTEBOOK_DIR / "DiffSplat"   # where the official repo is cloned
OUT_DIR       = NOTEBOOK_DIR / "out"          # checkpoints + generated outputs
DATA_DIR      = NOTEBOOK_DIR / "data"         # T3Bench prompts + GSO dataset

# ── Model choice ─────────────────────────────────────────────────────────────
# Options: "sd15" | "pas" | "sd35m"
MODEL = "sd15"

# ── PyTorch CUDA wheel channel ────────────────────────────────────────────────
# Match the CUDA version installed on your system.
# Common options: "cu121" (CUDA 12.1), "cu118" (CUDA 11.8)
TORCH_CHANNEL = "cu121"

# ── Optional Hugging Face mirror (uncomment if hf.co is slow) ────────────────
# HF_ENDPOINT = "https://hf-mirror.com"
HF_ENDPOINT = ""

# ── GPU id ───────────────────────────────────────────────────────────────────
GPU_ID = 0

# ── Random seed ──────────────────────────────────────────────────────────────
SEED = 0

print("Configuration:")
print(f"  NOTEBOOK_DIR  : {NOTEBOOK_DIR}")
print(f"  REPO_DIR      : {REPO_DIR}")
print(f"  OUT_DIR       : {OUT_DIR}")
print(f"  DATA_DIR      : {DATA_DIR}")
print(f"  MODEL         : {MODEL}")
print(f"  TORCH_CHANNEL : {TORCH_CHANNEL}")
print(f"  GPU_ID        : {GPU_ID}")

## 1 — Environment Setup

Clone the official DiffSplat repository and install all Python dependencies.

> **Note**: This only needs to be run once.  
> If `DiffSplat/` already exists the clone step is skipped automatically.

In [ ]:
import sys

# Make sure our wrapper modules are importable from any working directory
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from diffsplat_tools.setup import clone_repo, install_repo

# 1a. Clone the official repo (depth=1 for speed)
clone_repo(REPO_DIR, ref=None, dry_run=False)
print(f"\nRepo ready at: {REPO_DIR}")

In [ ]:
# 1b. Install PyTorch, xformers, and all DiffSplat requirements
# This can take several minutes on a fresh environment.
install_repo(
    REPO_DIR,
    torch_channel=TORCH_CHANNEL,
    skip_xformers=False,
    skip_rasterizer=False,
    dry_run=False,
)
print("\nInstallation complete.")

### Verify GPU availability

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"  GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM")

## 2 — Download Checkpoints

Pull pretrained weights for the selected model from Hugging Face.  
Both the **text-conditioned** and **image-conditioned** checkpoints are downloaded.

In [ ]:
from diffsplat_tools.common import command_env, DEFAULT_CACHE_ROOT
from diffsplat_tools.downloads import download_checkpoints

env = command_env(
    hf_home=str(DEFAULT_CACHE_ROOT / "huggingface"),
    torch_home=str(DEFAULT_CACHE_ROOT / "torch"),
    hf_endpoint=HF_ENDPOINT or None,
)

download_checkpoints(
    repo_dir=REPO_DIR,
    out_dir=OUT_DIR,
    model=MODEL,
    variant="both",   # "text" | "image" | "both"
    env=env,
    dry_run=False,
)
print(f"\nCheckpoints saved to: {OUT_DIR}")

## 3 — Download Benchmark Data

Download the **T3Bench** prompt list (small, fast) and the **GSO rendered** dataset (several GB, skippable).

In [ ]:
from diffsplat_tools.downloads import download_t3bench, download_gso

# T3Bench prompts (~10 KB)
download_t3bench(DATA_DIR, dry_run=False)
print(f"T3Bench prompts at: {DATA_DIR / 't3bench' / 't3bench_prompt.txt'}")

In [ ]:
# GSO rendered dataset (~several GB) — skip if you don't need image evaluation
SKIP_GSO = True   # Set to False to download

if not SKIP_GSO:
    download_gso(
        DATA_DIR,
        dry_run=False,
        hf_home=str(DEFAULT_CACHE_ROOT / "huggingface"),
        hf_endpoint=HF_ENDPOINT or None,
    )
    print(f"GSO dataset at: {DATA_DIR / 'gso_rendered'}")
else:
    print("GSO download skipped (SKIP_GSO=True).")

## 4 — Text-Conditioned Inference

Generate a 3D Gaussian splat from a **text prompt**.

The output is written to `OUT_DIR/<checkpoint_tag>/` as a GIF turntable and optionally a PLY file.

In [ ]:
import subprocess, sys

TEXT_PROMPT         = "a_toy_robot"   # underscores replace spaces
GUIDANCE_SCALE      = 7.5
NUM_TIMESTEPS       = 20
OUTPUT_VIDEO_TYPE   = "gif"           # "gif" or "mp4"

cmd = [
    sys.executable,
    str(NOTEBOOK_DIR / "scripts" / "run_text.py"),
    "--model",                MODEL,
    "--prompt",               TEXT_PROMPT,
    "--seed",                 str(SEED),
    "--gpu-id",               str(GPU_ID),
    "--guidance-scale",       str(GUIDANCE_SCALE),
    "--num-inference-timesteps", str(NUM_TIMESTEPS),
    "--output-video-type",    OUTPUT_VIDEO_TYPE,
    "--repo-dir",             str(REPO_DIR),
    "--out-dir",              str(OUT_DIR),
]

if HF_ENDPOINT:
    cmd += ["--hf-endpoint", HF_ENDPOINT]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(NOTEBOOK_DIR), check=True)

### Display the generated turntable GIF

In [ ]:
from IPython.display import Image as IPImage, display
from diffsplat_tools.constants import MODEL_SPECS
import glob

tag = MODEL_SPECS[MODEL].text_tag
pattern = str(OUT_DIR / tag / f"*{TEXT_PROMPT}*.gif")
gifs = sorted(glob.glob(pattern))

if gifs:
    print(f"Found {len(gifs)} GIF(s). Showing the most recent:")
    display(IPImage(filename=gifs[-1]))
else:
    print(f"No GIF found matching: {pattern}")
    # List all files in the output tag directory
    out_tag_dir = OUT_DIR / tag
    if out_tag_dir.exists():
        files = sorted(out_tag_dir.rglob("*"))
        print("Files in output directory:")
        for f in files[:20]:
            print(" ", f)

### Batch T3Bench text evaluation (optional)

Runs inference on all T3Bench prompts and computes CLIP / R-Precision / ImageReward metrics.  
This can take hours depending on GPU speed.

In [ ]:
RUN_T3BENCH_EVAL = False   # Set to True to run the full benchmark

if RUN_T3BENCH_EVAL:
    t3bench_file = DATA_DIR / "t3bench" / "t3bench_prompt.txt"
    cmd = [
        sys.executable,
        str(NOTEBOOK_DIR / "scripts" / "run_text.py"),
        "--model",        MODEL,
        "--prompt-file",  str(t3bench_file),
        "--eval",
        "--seed",         str(SEED),
        "--gpu-id",       str(GPU_ID),
        "--repo-dir",     str(REPO_DIR),
        "--out-dir",      str(OUT_DIR),
    ]
    subprocess.run(cmd, cwd=str(NOTEBOOK_DIR), check=True)
else:
    print("T3Bench batch eval skipped (RUN_T3BENCH_EVAL=False).")

## 5 — Image-Conditioned Inference

Reconstruct a 3D Gaussian splat given a **single reference image**.  
The bundled `frog.png` asset is used as the default example.

In [ ]:
from IPython.display import Image as IPImage, display

IMAGE_PATH       = REPO_DIR / "assets" / "grm" / "frog.png"
IMAGE_PROMPT     = "a_frog"
ELEVATION        = 20.0
IMG_GUIDANCE     = 2.0
IMG_TIMESTEPS    = 20
REMBG_CENTER     = True    # remove background and center the object
TRIANGLE_CFG     = True    # apply triangle CFG scaling

# Preview the input image
if IMAGE_PATH.exists():
    display(IPImage(filename=str(IMAGE_PATH), width=256))
else:
    print(f"Input image not found: {IMAGE_PATH}")
    print("Set IMAGE_PATH to an existing PNG/JPG file.")

In [ ]:
cmd = [
    sys.executable,
    str(NOTEBOOK_DIR / "scripts" / "run_image.py"),
    "--model",                   MODEL,
    "--image-path",              str(IMAGE_PATH),
    "--prompt",                  IMAGE_PROMPT,
    "--elevation",               str(ELEVATION),
    "--guidance-scale",          str(IMG_GUIDANCE),
    "--num-inference-timesteps", str(IMG_TIMESTEPS),
    "--seed",                    str(SEED),
    "--gpu-id",                  str(GPU_ID),
    "--repo-dir",                str(REPO_DIR),
    "--out-dir",                 str(OUT_DIR),
]
if REMBG_CENTER:
    cmd.append("--rembg-and-center")
if TRIANGLE_CFG:
    cmd.append("--triangle-cfg-scaling")
if HF_ENDPOINT:
    cmd += ["--hf-endpoint", HF_ENDPOINT]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(NOTEBOOK_DIR), check=True)

### Display the image-conditioned result

In [ ]:
img_tag = MODEL_SPECS[MODEL].image_tag
pattern = str(OUT_DIR / img_tag / f"*{IMAGE_PROMPT}*.gif")
gifs = sorted(glob.glob(pattern))

if gifs:
    print(f"Found {len(gifs)} GIF(s). Showing the most recent:")
    display(IPImage(filename=gifs[-1]))
else:
    print(f"No GIF found matching: {pattern}")

## 6 — Image Quality Evaluation

Compute **PSNR**, **SSIM**, and **LPIPS** between predicted renders and ground-truth images.  
Requires that both `pred_dir` and `gt_dir` contain matching image filenames.

In [ ]:
import json
from diffsplat_tools.evaluation import evaluate_image_dirs

# ── Paths to adjust ──────────────────────────────────────────────────────────
# PRED_DIR should point to the rendered views produced by DiffSplat.
# GT_DIR   should point to the matching ground-truth views.

PRED_DIR = OUT_DIR / "predictions"    # example — change to your actual output
GT_DIR   = DATA_DIR / "gso_rendered"  # example — change to your GT directory
METRICS_JSON = NOTEBOOK_DIR / "results" / "metrics.json"
SKIP_LPIPS   = False                  # set True to skip LPIPS (slower, needs lpips pkg)

if not PRED_DIR.exists() or not GT_DIR.exists():
    print("PRED_DIR or GT_DIR not found — skipping evaluation.")
    print(f"  PRED_DIR : {PRED_DIR}  (exists: {PRED_DIR.exists()})")
    print(f"  GT_DIR   : {GT_DIR}  (exists: {GT_DIR.exists()})")
else:
    import argparse
    eval_args = argparse.Namespace(
        pred_dir=str(PRED_DIR),
        gt_dir=str(GT_DIR),
        json=str(METRICS_JSON),
        device="auto",
        skip_lpips=SKIP_LPIPS,
        save_per_image=True,
        limit=None,
    )
    evaluate_image_dirs(eval_args)

### Load and display saved metrics (if evaluation was run)

In [ ]:
if METRICS_JSON.exists():
    metrics = json.loads(METRICS_JSON.read_text())
    print(f"Images evaluated : {metrics['count']}")
    print(f"PSNR             : {metrics['psnr']['mean']:.2f} ± {metrics['psnr']['std']:.2f} dB")
    print(f"SSIM             : {metrics['ssim']['mean']:.4f} ± {metrics['ssim']['std']:.4f}")
    if 'lpips' in metrics:
        print(f"LPIPS            : {metrics['lpips']['mean']:.4f} ± {metrics['lpips']['std']:.4f}")
else:
    print("Metrics file not found — run the evaluation cell above first.")

## 7 — Ablation: Render-Loss Comparison

Run inference with both the **render** and **no-render** checkpoints to compare outputs.

> The no-render checkpoint is **not** in the public Hugging Face release.  
> Place it at `OUT_DIR/<no_render_tag>/` before running this cell.

In [ ]:
from diffsplat_tools.constants import MODEL_SPECS

spec = MODEL_SPECS[MODEL]
no_render_dir = OUT_DIR / spec.no_render_tag

print(f"Render checkpoint tag    : {spec.text_tag}")
print(f"No-render checkpoint tag : {spec.no_render_tag}")
print(f"No-render dir exists     : {no_render_dir.exists()}")

In [ ]:
RUN_RENDER_ABLATION = no_render_dir.exists()   # auto-skip when not available
ABLATION_PROMPT     = "a_toy_robot"

if RUN_RENDER_ABLATION:
    cmd = [
        sys.executable,
        str(NOTEBOOK_DIR / "scripts" / "ablate_render_loss.py"),
        "--model",    MODEL,
        "--prompt",   ABLATION_PROMPT,
        "--seed",     str(SEED),
        "--gpu-id",   str(GPU_ID),
        "--repo-dir", str(REPO_DIR),
        "--out-dir",  str(OUT_DIR),
    ]
    subprocess.run(cmd, cwd=str(NOTEBOOK_DIR), check=True)
else:
    print(f"Skipped: no-render checkpoint not found at {no_render_dir}.")
    print("See README.md — Ablation Note for details.")

### Side-by-side comparison

In [ ]:
if RUN_RENDER_ABLATION:
    render_gifs    = sorted((OUT_DIR / spec.text_tag).glob(f"*{ABLATION_PROMPT}*.gif"))
    no_render_gifs = sorted((OUT_DIR / spec.no_render_tag).glob(f"*{ABLATION_PROMPT}*.gif"))

    print("Render checkpoint output:")
    if render_gifs:
        display(IPImage(filename=str(render_gifs[-1])))
    else:
        print("  No GIF found.")

    print("No-render checkpoint output:")
    if no_render_gifs:
        display(IPImage(filename=str(no_render_gifs[-1])))
    else:
        print("  No GIF found.")

## 8 — Quick Reference

Equivalent **command-line** invocations for the operations above:

```bash
# Full one-shot setup
python3 scripts/prepare_public.py --model sd15 --skip-gso

# Text inference
python3 scripts/run_text.py --model sd15 --prompt a_toy_robot

# T3Bench batch eval
python3 scripts/run_text.py --model sd15 \
  --prompt-file data/t3bench/t3bench_prompt.txt --eval

# Image inference
python3 scripts/run_image.py --model sd15 \
  --image-path DiffSplat/assets/grm/frog.png \
  --prompt a_frog --elevation 20 \
  --rembg-and-center --triangle-cfg-scaling --guidance-scale 2

# Image metrics
python3 scripts/evaluate_image.py \
  --pred-dir out/predictions \
  --gt-dir   data/gso_rendered \
  --json     results/metrics.json

# Render-loss ablation
python3 scripts/ablate_render_loss.py --model sd15 --prompt a_toy_robot

# Pass extra upstream flags after --
python3 scripts/run_text.py --model sd15 --prompt a_toy_robot -- --save_ply
```

### Model variants

| Key | Backbone | VRAM | Notes |
|-----|----------|------|-------|
| `sd15` | Stable Diffusion 1.5 | ~16 GB | Fastest, default |
| `pas` | PixelArt-Sigma | ~16 GB | Higher fidelity textures |
| `sd35m` | Stable Diffusion 3.5 Medium | ~40 GB | Best quality |